# Mini-TP 6 — Mi modelo en el Data Lake (Sesión 6)

Subo **mi modelo** (`iris-rf`, el mismo de los Mini-TPs 1–4) a un Data Lake
compatible con S3 (**MinIO** en local) y lo **sirvo cargándolo desde allí**, no
desde disco.

El código vive en `src/model_service/datalake.py` (compartido, como el resto):
crea el bucket con **zonas** por prefijo (`raw`, `curated`, `models`), sube el
dataset y el artefacto **versionado** (`models/v1/…`) y lo descarga a memoria.
`model.py` ahora acepta `MODEL_SOURCE=lake`: la API REST del Mini-TP 1 arranca
y baja el modelo del lake en vez de leer `model/model.joblib`.

**Requisito:** MinIO en `localhost:9000` (usuario `minio` / `minio123`):

```bash
docker run -d --name minio -p 9000:9000 -p 9001:9001 \
  -e MINIO_ROOT_USER=minio -e MINIO_ROOT_PASSWORD=minio123 \
  quay.io/minio/minio server /data --console-address ":9001"
```

## 0. Configuración

Endpoint y credenciales por variables de entorno (contra S3 real sólo cambian estos valores).

In [1]:
import os, requests

os.environ.setdefault("AWS_ACCESS_KEY_ID", "minio")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "minio123")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

from model_service import datalake as dl

r = requests.get(f"{dl.ENDPOINT}/minio/health/live", timeout=3)
print("MinIO:", dl.ENDPOINT, "-> HTTP", r.status_code)
print("bucket:", dl.BUCKET)

MinIO: http://localhost:9000 -> HTTP 200
bucket: mlops2-lake


## 1. Bucket y zonas

S3 no tiene carpetas: cada zona es un **prefijo** del mismo bucket.

| Zona | Qué guarda | Quién escribe |
|---|---|---|
| `raw/` | el dato tal cual llega | ingesta |
| `curated/` | dato limpio, con el esquema del contrato | ETL |
| `models/` | artefactos versionados (`models/v1/`, `models/v2/`, …) | entrenamiento |

In [2]:
s3 = dl.client()
dl.create_zones(s3)

['s3://mlops2-lake/raw/',
 's3://mlops2-lake/curated/',
 's3://mlops2-lake/models/']

## 2. El modelo

El mismo artefacto de siempre (`train.py` → `model/model.joblib` con el pipeline y su metadata). Si no está, lo entreno.

In [3]:
from pathlib import Path
import subprocess, sys

if not Path("../model/model.joblib").exists():
    subprocess.run([sys.executable, "../train.py"], check=True)
print("artefacto local:", Path("../model/model.joblib").resolve())

artefacto local: /home/chris/Documents/MIA/mlops-2-tps-uba/model/model.joblib


## 3. Subir dataset y modelo al lake

- `raw/iris/iris.csv`: Iris como lo entrega sklearn (columnas `sepal length (cm)`, …).
- `curated/iris/iris.csv`: columnas renombradas a las features del contrato (`sepal_length_cm`, …) y sin duplicados.
- `models/v1/`: `model.joblib` + `metadata.json`.

In [4]:
print(dl.upload_dataset(s3))
print(dl.upload_model("v1", s3))

for key, size in dl.list_objects("", s3):
    print(f"{size:>8}  s3://{dl.BUCKET}/{key}")

{'raw': 'raw/iris/iris.csv', 'curated': 'curated/iris/iris.csv', 'filas_raw': 150, 'filas_curated': 149}
['models/v1/model.joblib', 'models/v1/metadata.json']
     579  s3://mlops2-lake/audit/serving/20261008T233706.jsonl
     579  s3://mlops2-lake/audit/serving/20261008T233804.jsonl
     579  s3://mlops2-lake/audit/serving/20261008T233822.jsonl
     579  s3://mlops2-lake/audit/serving/20261008T235719.jsonl
     579  s3://mlops2-lake/audit/serving/20261009T000949.jsonl
     579  s3://mlops2-lake/audit/serving/20261009T001013.jsonl
     579  s3://mlops2-lake/audit/serving/20261009T001402.jsonl
       0  s3://mlops2-lake/curated/.keep
    2751  s3://mlops2-lake/curated/iris/iris.csv
     731  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-fcd6277db6ad409faa001cb76df253e4/artifacts/MLmodel
     217  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-fcd6277db6ad409faa001cb76df253e4/artifacts/conda.yaml
 4748210  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-fcd6277db6ad409faa001cb76

In [5]:
print("versiones publicadas:", dl.list_versions(s3))
dl.read_metadata("v1", s3)

versiones publicadas: ['v1']


{'model_name': 'iris-rf',
 'model_version': '1.0.0',
 'algorithm': 'RandomForestClassifier',
 'dataset': 'sklearn.datasets.load_iris',
 'features': ['sepal_length_cm',
  'sepal_width_cm',
  'petal_length_cm',
  'petal_width_cm'],
 'target_names': ['setosa', 'versicolor', 'virginica'],
 'test_accuracy': 0.9,
 'test_f1': 0.8997,
 'test_auc': 0.9933,
 'trained_at': '2026-09-04T00:36:03.178930+00:00'}

## 4. Servir el modelo desde el lake

Primero la función pedida: `load_model` descarga `models/v1/model.joblib` **a
memoria** (sin escribir en disco) y predice.

In [6]:
artifact = dl.load_model("v1", s3)
pipeline, meta = artifact["pipeline"], artifact["metadata"]

x = [[5.1, 3.5, 1.4, 0.2], [6.7, 3.0, 5.2, 2.3]]
print([meta["target_names"][p] for p in pipeline.predict(x)])

['setosa', 'virginica']


Ahora el servicio real. Levanto la API REST del Mini-TP 1 en **otro proceso**
con `MODEL_SOURCE=lake` y `MODEL_VERSION=v1`: al arrancar, `model.py` baja el
artefacto del lake. `/health` informa de dónde salió el modelo.

In [7]:
import time

env = {**os.environ, "MODEL_SOURCE": "lake", "MODEL_VERSION": "v1"}
api = subprocess.Popen(
    [sys.executable, "-m", "uvicorn", "model_service.rest:app", "--port", "8160", "--log-level", "warning"],
    env=env,
)
BASE = "http://127.0.0.1:8160"
for _ in range(50):
    try:
        requests.get(f"{BASE}/health", timeout=0.5); break
    except requests.ConnectionError:
        time.sleep(0.2)

requests.get(f"{BASE}/health").json()

{'status': 'ok',
 'model_loaded': True,
 'model_version': '1.0.0',
 'model_source': 'lake'}

In [8]:
ok = {"sepal_length_cm": 6.7, "sepal_width_cm": 3.0,
      "petal_length_cm": 5.2, "petal_width_cm": 2.3}
requests.post(f"{BASE}/v1/predict", json=ok).json()

{'prediction': 2, 'prediction_label': 'virginica', 'model_version': '1.0.0'}

In [9]:
api.terminate(); api.wait()
print("API detenida")

API detenida


## 5. (Opcional) MLflow apuntando a MinIO

El *tracking* (parámetros y métricas) va a un SQLite local; los **artefactos**
del experimento van al lake, bajo `s3://mlops2-lake/mlflow/`. MLflow usa el
mismo `boto3`, sólo necesita `MLFLOW_S3_ENDPOINT_URL`.

In [10]:
import tempfile, mlflow, mlflow.sklearn

os.environ["MLFLOW_S3_ENDPOINT_URL"] = dl.ENDPOINT
tracking_dir = tempfile.mkdtemp(prefix="mlflow-tp6-")
mlflow.set_tracking_uri(f"sqlite:///{tracking_dir}/mlflow.db")

EXP = "iris-rf-datalake"
if mlflow.get_experiment_by_name(EXP) is None:
    mlflow.create_experiment(EXP, artifact_location=f"s3://{dl.BUCKET}/mlflow/{EXP}")
mlflow.set_experiment(EXP)

with mlflow.start_run(run_name="iris-rf-v1") as run:
    mlflow.log_params({"algorithm": meta["algorithm"], "model_version": meta["model_version"],
                       "lake_key": f"s3://{dl.BUCKET}/models/v1/model.joblib"})
    mlflow.log_metrics({"accuracy": meta["test_accuracy"], "f1": meta["test_f1"], "auc": meta["test_auc"]})
    # MLflow 3 serializa con skops (no pickle) y exige declarar qué tipos se confían
    # al cargar; el árbol de sklearn es nuestro, así que lo marco como confiable.
    mlflow.sklearn.log_model(pipeline, name="model", skops_trusted_types=["sklearn.tree._tree.Tree"])

print("run:", run.info.run_id)
print("artifact_uri:", run.info.artifact_uri)

2026/10/08 20:17:11 INFO mlflow.store.db.utils: Creating initial MLflow database tables...


2026/10/08 20:17:11 INFO mlflow.store.db.utils: Updating database tables


2026/10/08 20:17:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


run: 44f6bc61778147579e84d779b77766ca
artifact_uri: s3://mlops2-lake/mlflow/iris-rf-datalake/44f6bc61778147579e84d779b77766ca/artifacts


In [11]:
for key, size in dl.list_objects("mlflow/", s3)[:10]:
    print(f"{size:>8}  s3://{dl.BUCKET}/{key}")

     731  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-e16c439ac00a4e999e79ff3e6f6d1c51/artifacts/MLmodel
     217  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-e16c439ac00a4e999e79ff3e6f6d1c51/artifacts/conda.yaml
 4748210  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-e16c439ac00a4e999e79ff3e6f6d1c51/artifacts/model.skops
     107  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-e16c439ac00a4e999e79ff3e6f6d1c51/artifacts/python_env.yaml
     103  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-e16c439ac00a4e999e79ff3e6f6d1c51/artifacts/requirements.txt
     731  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-fcd6277db6ad409faa001cb76df253e4/artifacts/MLmodel
     217  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-fcd6277db6ad409faa001cb76df253e4/artifacts/conda.yaml
 4748210  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-fcd6277db6ad409faa001cb76df253e4/artifacts/model.skops
     107  s3://mlops2-lake/mlflow/iris-rf-datalake/models/m-fcd6277db6ad409faa001

## 6. Reflexión

**1. ¿Por qué cargar el modelo desde el lake y no hornearlo en la imagen?**
Separa el ciclo de vida del modelo del de la aplicación. Con el modelo dentro de
la imagen, cada reentrenamiento obliga a reconstruir y redesplegar el contenedor
aunque el código no cambie. Con el lake, la misma imagen sirve cualquier versión
cambiando `MODEL_VERSION`. Además el artefacto queda en un único lugar
auditable, la imagen es más chica y varias réplicas (o REST y gRPC a la vez)
leen exactamente el mismo binario.

**2. ¿Cómo versionar un modelo nuevo sin romper el de producción?**
Publicarlo en un prefijo nuevo e inmutable (`models/v2/`) sin tocar `v1/`.
Producción sigue fijada a `MODEL_VERSION=v1`. La `v2` se prueba en una réplica
aparte (shadow o canary) y, si anda bien, se cambia el puntero a `v2`. El
rollback es volver el puntero a `v1`. En MLflow lo mismo se hace con un alias
(`champion`) en el Model Registry.

**3. ¿En qué zona guardar las predicciones que expone GraphQL?**
En `curated/` (por ejemplo, `curated/predictions/fecha=AAAA-MM-DD/`).
Son datos ya validados y con un esquema fijo (input, clase, versión del modelo,
timestamp), listos para que GraphQL los consulte sin limpiarlos. Si se quisiera
guardar también el evento crudo tal como llegó, eso iría a `raw/`, y `curated/`
quedaría como la vista que se consulta.